# Diebold–Mariano (DM) Test in Python

A reusable implementation that compares two sets of out-of-sample forecasts, calculates the loss differential, the DM statistic, the p-value, and automatically identifies the better-performing model.

**Key idea:** the DM test doesn't compare RMSE/MAE values directly — it tests whether the *average loss differential* between two models' forecast errors is statistically different from zero.

- **H0:** E[dₜ] = 0 (the two models have equal predictive accuracy)
- **H1:** E[dₜ] ≠ 0 (the two models have different predictive accuracy)

where dₜ = L_{A,t} − L_{B,t} is the loss differential at each forecast point.

## 1. The `diebold_mariano_test` Function

This function takes the actual values plus forecasts from two models, computes the loss differential (squared or absolute error), the DM statistic, and reports which model is better when the difference is significant.

In [ ]:
import numpy as np
from scipy import stats


def diebold_mariano_test(
    actual,
    forecast_a,
    forecast_b,
    significance=0.05,
    loss="squared"
):
    """
    Perform the Diebold-Mariano test.

    Parameters
    ----------
    actual : array-like
        Actual observed values.

    forecast_a : array-like
        Forecasts from Model A.

    forecast_b : array-like
        Forecasts from Model B.

    significance : float
        Significance level, e.g. 0.05.

    loss : str
        Loss function:
        "squared" -> Squared Error
        "absolute" -> Absolute Error

    Returns
    -------
    dict
        DM statistic, p-value, mean loss differential,
        decision and conclusion.
    """

    actual = np.asarray(actual, dtype=float)
    forecast_a = np.asarray(forecast_a, dtype=float)
    forecast_b = np.asarray(forecast_b, dtype=float)

    # Check lengths
    if not (
        len(actual) == len(forecast_a) == len(forecast_b)
    ):
        raise ValueError(
            "actual, forecast_a and forecast_b "
            "must have the same length."
        )

    # Remove missing values
    valid = (
        np.isfinite(actual)
        & np.isfinite(forecast_a)
        & np.isfinite(forecast_b)
    )

    actual = actual[valid]
    forecast_a = forecast_a[valid]
    forecast_b = forecast_b[valid]

    # Forecast errors
    error_a = actual - forecast_a
    error_b = actual - forecast_b

    # Calculate losses
    if loss == "squared":
        loss_a = error_a ** 2
        loss_b = error_b ** 2

    elif loss == "absolute":
        loss_a = np.abs(error_a)
        loss_b = np.abs(error_b)

    else:
        raise ValueError(
            "loss must be 'squared' or 'absolute'"
        )

    # Loss differential
    # Positive -> Model B has lower loss
    # Negative -> Model A has lower loss
    loss_diff = loss_a - loss_b

    n = len(loss_diff)

    # Mean loss differential
    mean_loss_diff = np.mean(loss_diff)

    # Variance of loss differential
    variance = np.var(
        loss_diff,
        ddof=1
    )

    if variance == 0:
        raise ValueError(
            "Variance of loss differential is zero."
        )

    # DM statistic
    dm_statistic = (
        mean_loss_diff
        / np.sqrt(variance / n)
    )

    # Two-sided p-value
    p_value = 2 * stats.norm.sf(
        abs(dm_statistic)
    )

    # Decision
    if p_value <= significance:

        decision = "Reject H0"

        if mean_loss_diff > 0:
            conclusion = (
                "Model B has significantly better "
                "forecast accuracy than Model A."
            )
            better_model = "Model B"

        else:
            conclusion = (
                "Model A has significantly better "
                "forecast accuracy than Model B."
            )
            better_model = "Model A"

    else:

        decision = "Fail to Reject H0"

        conclusion = (
            "There is insufficient evidence that "
            "the two models have different "
            "forecast accuracy."
        )

        better_model = "No statistically significant winner"

    # Display results
    print("=" * 70)
    print("DIEBOLD-MARIANO TEST")
    print("=" * 70)

    print(f"Number of forecasts : {n}")
    print(f"Loss function       : {loss}")
    print(f"Significance level  : {significance}")

    print("\nHypotheses:")
    print("H0: Models have equal predictive accuracy.")
    print("H1: Models have different predictive accuracy.")

    print("\nResults:")
    print(f"Mean Loss - Model A : {np.mean(loss_a):.6f}")
    print(f"Mean Loss - Model B : {np.mean(loss_b):.6f}")
    print(f"Mean Loss Difference: {mean_loss_diff:.6f}")
    print(f"DM Statistic        : {dm_statistic:.4f}")
    print(f"p-value             : {p_value:.6f}")

    print("\nDecision:")
    print(f"{decision}")

    print("\nConclusion:")
    print(conclusion)

    print(f"\nBetter Model:")
    print(better_model)

    print("=" * 70)

    return {
        "dm_statistic": dm_statistic,
        "p_value": p_value,
        "mean_loss_difference": mean_loss_diff,
        "mean_loss_model_a": np.mean(loss_a),
        "mean_loss_model_b": np.mean(loss_b),
        "decision": decision,
        "conclusion": conclusion,
        "better_model": better_model
    }

## 2. Example With Two Forecasting Models

Suppose you have actual values and predictions from ARIMA (Model A) and LSTM (Model B).

In [ ]:
np.random.seed(42)

# Actual observations
actual = np.array([
    101, 103, 102, 105, 107,
    106, 110, 112, 111, 115,
    117, 116, 120, 122, 121
])

# Model A predictions
forecast_a = np.array([
    100, 102, 103, 104, 106,
    105, 108, 111, 110, 113,
    116, 115, 118, 121, 120
])

# Model B predictions
forecast_b = np.array([
    101, 102, 102, 105, 106,
    107, 109, 112, 111, 114,
    117, 116, 120, 122, 121
])

In [ ]:
result = diebold_mariano_test(
    actual=actual,
    forecast_a=forecast_a,
    forecast_b=forecast_b,
    significance=0.05,
    loss="squared"
)

## 3. Using MAE-Type Loss Instead

You can use absolute error instead of squared error:

| Loss | Formula | Similar to |
|---|---|---|
| `"squared"` | (error)² | RMSE / MSE evaluation |
| `"absolute"` | \|error\| | MAE evaluation |

In [ ]:
result = diebold_mariano_test(
    actual,
    forecast_a,
    forecast_b,
    significance=0.05,
    loss="absolute"
)

## 4. Understanding the Decision

The DM test hypotheses are:

$$H_0: \text{Equal predictive accuracy}$$
$$H_1: \text{Different predictive accuracy}$$

At α = 0.05, the decision is:

| p-value | Decision | Interpretation |
|---|---|---|
| p ≤ 0.05 | Reject H0 | Significant difference in forecast accuracy |
| p > 0.05 | Fail to reject H0 | No significant difference detected |

But when you reject H0, you still need to identify which model is better. That's why the function also calculates `mean_loss_difference`, where:

$$d_t = L_{A,t} - L_{B,t}$$

- If d̄ > 0 &rarr; L̄_A > L̄_B &rarr; **Model B is better**
- If d̄ < 0 &rarr; L̄_A < L̄_B &rarr; **Model A is better**

## 5. Example Interpretation

Suppose the squared-error run above produced output like:

```
Mean Loss - Model A : 0.084521
Mean Loss - Model B : 0.061238
Mean Loss Difference: 0.023283
DM Statistic        : 2.3145
p-value             : 0.0206

Decision:
Reject H0

Conclusion:
Model B has significantly better forecast accuracy than Model A.
```

Because 0.0206 < 0.05, we reject H0. And since 0.023283 > 0, Model A has the larger loss.

**Therefore:** Model B is statistically significantly more accurate than Model A under the selected loss function.

## 6. For a Stock-Price Forecasting Project

A typical application:

```
Historical Stock Data
        |
Train / Validation / Test
        |
 +--------------+
 |              |
ARIMA          LSTM
 |              |
 v              v
Forecast A    Forecast B
 |              |
 +------+-------+
        v
 Actual vs Forecast
        v
 Forecast Errors
        v
 Loss Differential
        v
 Diebold-Mariano Test
        v
 +--------+---------+
 |                  |
p > 0.05         p <= 0.05
 |                  |
 v                  v
No significant    Significant
difference        difference
                     |
                     v
             Select lower-loss
                 model
```

The example below simulates an out-of-sample comparison between two simple forecasting strategies applied to a synthetic price series.

In [ ]:
np.random.seed(7)

# Simulate a simple synthetic price series
n_obs = 200
price = 100 + np.cumsum(np.random.normal(0, 1, n_obs))

# Split into train/test (out-of-sample)
train, test = price[:150], price[150:]

# "Model A": naive forecast (last observed value)
forecast_a_stock = np.roll(test, 1)
forecast_a_stock[0] = train[-1]

# "Model B": simple moving-average forecast (last 5 obs of train+test)
window = 5
combined = np.concatenate([train[-window:], test])
forecast_b_stock = np.array([
    combined[i:i + window].mean()
    for i in range(len(test))
])

result_stock = diebold_mariano_test(
    actual=test,
    forecast_a=forecast_a_stock,
    forecast_b=forecast_b_stock,
    significance=0.05,
    loss="squared"
)

## Important: Use Out-of-Sample Forecasts

Use out-of-sample forecasts for the DM test:

```
actual_test
forecast_arima
forecast_lstm
```

rather than comparing training/in-sample predictions.

> **Technical caveat:** the simple implementation above uses the basic DM variance estimate. For multi-step forecasts, overlapping forecast horizons, or strongly autocorrelated loss differentials, a production implementation should use an appropriate HAC/Newey–West long-run variance estimator (and often a small-sample correction).